# 🧪 04 · Fine-Tune a Model on QA Data (LoRA) — Experts

**Level:** Experts · **Time:** ~45–60 min · **Cost:** $0 (Colab free T4 GPU)

Now the real thing: **train** a model so it absorbs *our team's* test-case style from examples,
using **LoRA** (Low-Rank Adaptation) — the standard, cheap way to fine-tune. We train tiny
'adapter' weights instead of the whole model, so it fits on a free GPU.

> ⚙️ **Required:** *Runtime → Change runtime type → T4 GPU*.

### Step 1 — Install the training stack

In [ ]:
!pip install -q -U transformers datasets peft trl accelerate bitsandbytes
print('✅ training libs ready')

### Step 2 — Get the training data (downloads automatically)
This cell **downloads the sample QA dataset straight from the repo** — no manual upload needed.
Each line is one training example (requirement → ideal test cases) in chat format.

In [ ]:
import urllib.request
from datasets import load_dataset

URL = 'https://raw.githubusercontent.com/hn-alchemist/qa-ai-workshop/main/data/qa_finetune_dataset.jsonl'
PATH = 'qa_finetune_dataset.jsonl'
urllib.request.urlretrieve(URL, PATH)   # downloads into this Colab session
ds = load_dataset('json', data_files=PATH, split='train')
print(f'✅ downloaded and loaded {len(ds)} examples'); print(ds[0])

### Step 3 — Load a small base model (4-bit, fits free GPU)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE = 'Qwen/Qwen2.5-0.5B-Instruct'  # small = fast to train on a free GPU
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                         bnb_4bit_quant_type='nf4')
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map='auto')
print('✅ base model loaded')

### Step 4 — Configure LoRA + train
We only train small adapter matrices. A few hundred steps is enough to *see* the effect.
Bump `max_steps`/dataset size for real results.

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

peft_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                      task_type='CAUSAL_LM',
                      target_modules=['q_proj','k_proj','v_proj','o_proj'])

def to_text(ex):
    # ex['messages'] is a list of {role, content}; render with the chat template
    return {'text': tok.apply_chat_template(ex['messages'], tokenize=False)}
ds_text = ds.map(to_text)

args = SFTConfig(output_dir='qa-lora', per_device_train_batch_size=1,
                 gradient_accumulation_steps=4, learning_rate=2e-4,
                 max_steps=60, logging_steps=10, dataset_text_field='text',
                 max_seq_length=1024, report_to='none')
trainer = SFTTrainer(model=model, train_dataset=ds_text, args=args, peft_config=peft_cfg)
trainer.train()
print('✅ training done')

### Step 5 — Save the adapter & try it

In [ ]:
trainer.save_model('qa-lora')
print('✅ adapter saved to ./qa-lora  (download it from the Files panel)')

from transformers import pipeline
tuned = pipeline('text-generation', model=trainer.model, tokenizer=tok)
msgs = [{'role':'user','content':'Generate test cases for: user can update their email address.'}]
print(tuned(msgs, max_new_tokens=500, do_sample=False)[0]['generated_text'][-1]['content'])

### Step 6 (optional) — Publish to Hugging Face Hub (free)
Make a free account at huggingface.co, create a token (Settings → Access Tokens),
then your whole community can load the adapter by name.

In [ ]:
# from huggingface_hub import login; login()  # paste your free token
# trainer.model.push_to_hub('your-username/qa-lora')
print('Uncomment above to publish your fine-tuned adapter for the community.')

---
### 🧠 Expert discussion
- **LoRA vs full fine-tune:** we trained <1% of params — that's why it's free-GPU friendly.
- **Data is everything:** 12 toy examples won't transform the model. The real lesson is the
  *pipeline*. For production, gather 200–2000 high-quality `requirement → test cases` pairs.
- Compare base vs tuned on the *same* prompt — discuss what actually changed and why.
- Where would a great prompt (notebook 03) have been enough instead of training?

---
### 🎮 Try this (prove it worked)
From the [playground challenges](https://github.com/hn-alchemist/qa-ai-workshop/blob/main/data/playground_challenges.md) (challenge 13):
1. Generate test cases for the same requirement with the **base** model and your **fine-tuned** model.
2. Open **Notebook 07** and let the AI judge **score both** — did training actually help, with numbers?

Discuss: was the improvement worth the effort, or would a better prompt have done the job?